Importing libraries

In [ ]:
import random

import torch
from torch.utils.data import DataLoader, Dataset
from torch import nn

import pathlib as pl
import json

Handling Directories

In [ ]:
root_dir = pl.Path("data")
dataset_file = root_dir / "iraqi_support_dataset_3000.json"

dataset_file.is_file()  # checking if file exists

In [ ]:
with open(dataset_file) as file:
    test_data = json.load(file)

# printing first sample
test_data[random.randint(0, len(test_data))]

 Defining Acoustic Device

In [ ]:
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.mps.is_available() else "cpu")

device

### Modeling Pipeline

Categories

In [ ]:
CATEGORY_BY_CODE = {
    "A": "failed_transfer",
    "B": "wrong_recipient",
    "C": "payment_pending",
    "D": "login_problem",
    "E": "card_issue",
    "F": "account_issue",
    "G": "agent_dispute",
    "H": "other",
}
categories = tuple(CATEGORY_BY_CODE.values())
priorities = ("high", "medium", "low")
entity_fields = ("amount", "transaction_id", "recipient", "account_id", "date")
CONFIDENCE_THRESHOLD = 0.75  # Provisional; select the final value on validation data.
CATEGORY_LABEL_MASS_THRESHOLD = 0.50  # Provisional prompt-adherence guard.
MAX_NEW_TOKENS = 256

In [ ]:
# Importing Transformers

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16
)

In [ ]:
#model to device
model = model.to(device)

Prompt builder

- as system
- as user

In [ ]:
print(type(tokenizer))
print(type(model))

In [ ]:
def sys_prompt_build(categories: list) -> str:
    categories_text = "\n".join(f"-{c}" for c in categories)

    return f"""You are an Iraqi Arabic customer support ticket analyzer.
Analyze the full conversation.

Your tasks:
1. Classify the ticket into exactly one of the allowed categories.
2. Assign exactly one priority: high, medium, or low.
3. Extract only information explicitly mentioned in the conversation.
4. Do not guess missing values.
5. Return valid JSON only.

Allowed categories:
{categories_text}

Extract these fields:
- amount
- transaction_id
- recipient
- account_id
- date

Return JSON using exactly this structure:

{{
  "category": "category_name",
  "priority": "high | medium | low",
  "entities": {{
    "amount": null,
    "transaction_id": null,
    "recipient": null,
    "account_id": null,
    "date": null
  }}
}}
    """


def users_prompt_build(messages: list) -> str:
    return "\n".join(f"Customer: {m["text"]}" for m in messages)

In [ ]:
def tokenize(conv: list[dict]):
    text = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
    return tokenizer(text, return_tensors="pt").to(device)

In [ ]:
def build_conversation(complain: list, categories: list) -> list:
    return [
        {
            "role": "system",
            "content": sys_prompt_build(categories=categories)
        },
        {
            "role": "user",
            "content": users_prompt_build(complain)
        }
    ]

In [ ]:
def category_confidence_prompt_build() -> str:
    labels = "\n".join(
        f"{code} = {category}" for code, category in CATEGORY_BY_CODE.items()
    )
    return f"""Classify the complete Iraqi Arabic support conversation.
Return exactly one label and no other text.

{labels}
"""


def build_category_confidence_conversation(messages: list) -> list:
    return [
        {"role": "system", "content": category_confidence_prompt_build()},
        {"role": "user", "content": users_prompt_build(messages)},
    ]


def _candidate_sequence_log_probability(inputs, candidate_token_ids: list[int]) -> float:
    candidate = torch.tensor(
        [candidate_token_ids],
        dtype=inputs["input_ids"].dtype,
        device=inputs["input_ids"].device,
    )
    input_ids = torch.cat([inputs["input_ids"], candidate], dim=1)
    attention_mask = torch.cat(
        [inputs["attention_mask"], torch.ones_like(candidate)],
        dim=1,
    )
    prompt_length = inputs["input_ids"].shape[1]
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    candidate_logits = outputs.logits[
        :, prompt_length - 1:prompt_length + candidate.shape[1] - 1, :
    ]
    log_probabilities = torch.log_softmax(candidate_logits.float(), dim=-1)
    return log_probabilities.gather(-1, candidate.unsqueeze(-1)).sum().item()


def score_category_candidates(messages: list) -> dict:
    conversation = build_category_confidence_conversation(messages)
    inputs = tokenize(conv=conversation)
    codes = tuple(CATEGORY_BY_CODE)
    candidate_tokens = {
        code: tokenizer.encode(code, add_special_tokens=False) for code in codes
    }

    with torch.inference_mode():
        if all(len(token_ids) == 1 for token_ids in candidate_tokens.values()):
            next_token_logits = model(**inputs).logits[0, -1].float()
            token_ids = torch.tensor(
                [candidate_tokens[code][0] for code in codes],
                device=next_token_logits.device,
            )
            candidate_scores = next_token_logits[token_ids]
            label_mass = torch.softmax(next_token_logits, dim=-1)[token_ids].sum().item()
        else:
            candidate_scores = torch.tensor(
                [
                    _candidate_sequence_log_probability(inputs, candidate_tokens[code])
                    for code in codes
                ],
                device=inputs["input_ids"].device,
            )
            label_mass = torch.exp(candidate_scores).sum().item()

    probabilities = torch.softmax(candidate_scores, dim=0).cpu().tolist()
    best_index = max(range(len(codes)), key=probabilities.__getitem__)
    best_code = codes[best_index]
    return {
        "label": best_code,
        "category": CATEGORY_BY_CODE[best_code],
        "confidence": probabilities[best_index],
        "label_mass": label_mass,
        "probabilities": {
            CATEGORY_BY_CODE[code]: probability
            for code, probability in zip(codes, probabilities)
        },
    }

In [ ]:
complain = test_data[0]["messages"]
conversation = build_conversation(complain=complain, categories=categories)

In [ ]:
inputs = tokenize(conv=conversation)
inputs  # printing tokenized conv

Generate LLM Output

In [ ]:
def generate_llm_output(inputs):
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

In [ ]:
class ModelOutputError(ValueError):
    def __init__(self, code: str, message: str):
        super().__init__(message)
        self.code = code


def _clean_json_response(response: str) -> str:
    cleaned = response.strip()
    if cleaned.startswith("```") and cleaned.endswith("```"):
        lines = cleaned.splitlines()
        if len(lines) >= 3 and lines[0].lower() in {"```", "```json"}:
            cleaned = "\n".join(lines[1:-1]).strip()
    return cleaned


def parse_model_output(response: str) -> dict:
    try:
        parsed = json.loads(_clean_json_response(response))
    except (json.JSONDecodeError, TypeError, RecursionError) as error:
        raise ModelOutputError("malformed_json", str(error)) from error

    required_fields = {"category", "priority", "entities"}
    if not isinstance(parsed, dict) or set(parsed) != required_fields:
        raise ModelOutputError(
            "invalid_top_level_schema",
            f"Expected exactly {sorted(required_fields)}",
        )
    if parsed["category"] not in categories:
        raise ModelOutputError(
            "unsupported_category",
            f"Unsupported category: {parsed['category']!r}",
        )
    if parsed["priority"] not in priorities:
        raise ModelOutputError(
            "unsupported_priority",
            f"Unsupported priority: {parsed['priority']!r}",
        )

    entities = parsed["entities"]
    if not isinstance(entities, dict) or set(entities) != set(entity_fields):
        raise ModelOutputError(
            "invalid_entity_schema",
            f"Expected exactly {sorted(entity_fields)}",
        )
    amount = entities["amount"]
    if amount is not None and (not isinstance(amount, int) or isinstance(amount, bool)):
        raise ModelOutputError("invalid_amount", "amount must be an integer or null")
    for field in entity_fields[1:]:
        value = entities[field]
        if value is not None and (not isinstance(value, str) or not value.strip()):
            raise ModelOutputError(
                "invalid_entity_value",
                f"{field} must be a non-empty string or null",
            )

    return {
        "category": parsed["category"],
        "priority": parsed["priority"],
        "entities": {field: entities[field] for field in entity_fields},
    }


def generate_validated_output(conversation: list[dict]) -> dict:
    current_conversation = conversation
    responses = []
    last_error = None

    for attempt in range(2):
        response = generate_llm_output(tokenize(conv=current_conversation))
        responses.append(response)
        try:
            return {
                "prediction": parse_model_output(response),
                "escalated": False,
                "attempts": attempt + 1,
            }
        except ModelOutputError as error:
            last_error = error
            if attempt == 0:
                current_conversation = conversation + [
                    {"role": "assistant", "content": response},
                    {
                        "role": "user",
                        "content": (
                            f"The previous output was invalid ({error.code}). "
                            "Return one corrected JSON object using the required schema only."
                        ),
                    },
                ]

    return {
        "prediction": None,
        "escalated": True,
        "escalation_reason": last_error.code,
        "validation_error": str(last_error),
        "attempts": len(responses),
    }


def predict_ticket(messages: list) -> dict:
    category_result = score_category_candidates(messages)
    conversation = build_conversation(messages, categories)
    generation_result = generate_validated_output(conversation)

    if generation_result["escalated"]:
        return {
            "category": category_result["category"],
            "priority": None,
            "entities": None,
            "confidence": category_result["confidence"],
            "label_mass": category_result["label_mass"],
            "category_probabilities": category_result["probabilities"],
            "escalated": True,
            "escalation_reason": generation_result["escalation_reason"],
            "validation_error": generation_result["validation_error"],
            "attempts": generation_result["attempts"],
        }

    prediction = generation_result["prediction"]
    if prediction["category"] != category_result["category"]:
        return {
            **prediction,
            "category": category_result["category"],
            "generated_category": prediction["category"],
            "confidence": category_result["confidence"],
            "label_mass": category_result["label_mass"],
            "category_probabilities": category_result["probabilities"],
            "escalated": True,
            "escalation_reason": "category_generation_conflict",
            "attempts": generation_result["attempts"],
        }

    if category_result["label_mass"] < CATEGORY_LABEL_MASS_THRESHOLD:
        return {
            **prediction,
            "confidence": category_result["confidence"],
            "label_mass": category_result["label_mass"],
            "category_probabilities": category_result["probabilities"],
            "escalated": True,
            "escalation_reason": "category_label_mass_low",
            "attempts": generation_result["attempts"],
        }

    if category_result["confidence"] < CONFIDENCE_THRESHOLD:
        return {
            **prediction,
            "confidence": category_result["confidence"],
            "label_mass": category_result["label_mass"],
            "category_probabilities": category_result["probabilities"],
            "escalated": True,
            "escalation_reason": "classification_confidence_below_threshold",
            "attempts": generation_result["attempts"],
        }

    return {
        **prediction,
        "confidence": category_result["confidence"],
        "label_mass": category_result["label_mass"],
        "category_probabilities": category_result["probabilities"],
        "escalated": False,
        "escalation_reason": None,
        "attempts": generation_result["attempts"],
    }

In [ ]:
sample_idx = random.randint(0, len(test_data) - 1)
sample = test_data[sample_idx]  # getting messages of complain
print(f"testing sample idx {sample_idx}, msg_id:{sample["id"]}")

for m in sample["messages"]:
    print(f"Message Content: {m["text"]}")

prediction_result = predict_ticket(sample["messages"])

In [ ]:
prediction_result